In [2]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import preprocessing
# label_encoder object knows how to understand word labels.
label_encoder = preprocessing.LabelEncoder()
  

%matplotlib inline
np.random.seed(42)

In [4]:
#load dataset
df = pd.read_csv("data/maternal-health-risk.csv")

In [6]:
#view fist five rows of the dataset
df.head()

,Age,Systolic BP,Diastolic,BS,Body Temp,BMI,Previous Complications,Preexisting Diabetes,Gestational Diabetes,Mental Health,Heart Rate,Risk Level
0,22,90.0,60.0,9.0,100,18.0,1.0,1.0,0,1,80.0,High
1,22,110.0,70.0,7.1,98,20.4,0.0,0.0,0,0,74.0,Low
2,27,110.0,70.0,7.5,98,23.0,1.0,0.0,0,0,72.0,Low
3,20,100.0,70.0,7.2,98,21.2,0.0,0.0,0,0,74.0,Low
4,20,90.0,60.0,7.5,98,19.7,0.0,0.0,0,0,74.0,Low


In [12]:
df.describe()


,Age,Systolic BP,Diastolic,BS,Body Temp,BMI,Previous Complications,Preexisting Diabetes,Gestational Diabetes,Mental Health,Heart Rate
count,1205.000000,1200.000000,1201.000000,1203.000000,1205.000000,1187.000000,1203.000000,1203.000000,1205.000000,1205.00000,1203.000000
mean,27.482988,116.819167,77.166528,7.501064,98.395851,23.315080,0.175395,0.288446,0.117842,0.33444,75.817124
std,9.196765,18.715502,14.305148,3.049522,1.088363,3.875682,0.380463,0.453228,0.322555,0.47199,7.227338
min,10.000000,70.000000,40.000000,3.000000,97.000000,0.000000,0.000000,0.000000,0.000000,0.00000,58.000000
25%,21.000000,100.000000,65.000000,6.000000,98.000000,20.450000,0.000000,0.000000,0.000000,0.00000,70.000000
50%,25.000000,120.000000,80.000000,6.900000,98.000000,23.000000,0.000000,0.000000,0.000000,0.00000,76.000000
75%,31.000000,130.000000,90.000000,7.900000,98.000000,25.000000,0.000000,1.000000,0.000000,1.00000,80.000000
max,65.000000,200.000000,140.000000,19.000000,103.000000,37.000000,1.000000,1.000000,1.000000,1.00000,92.000000


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1205 entries, 0 to 1204
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     1205 non-null   int64  
 1   Systolic BP             1200 non-null   float64
 2   Diastolic               1201 non-null   float64
 3   BS                      1203 non-null   float64
 4   Body Temp               1205 non-null   int64  
 5   BMI                     1187 non-null   float64
 6   Previous Complications  1203 non-null   float64
 7   Preexisting Diabetes    1203 non-null   float64
 8   Gestational Diabetes    1205 non-null   int64  
 9   Mental Health           1205 non-null   int64  
 10  Heart Rate              1203 non-null   float64
 11  Risk Level              1187 non-null   object 
dtypes: float64(7), int64(4), object(1)
memory usage: 113.1+ KB


# Data Cleaning <a class="anchor" id="cleaning"></a>


In [10]:
#check for missing values
df.isnull().sum()

Age                        0
Systolic BP                5
Diastolic                  4
BS                         2
Body Temp                  0
BMI                       18
Previous Complications     2
Preexisting Diabetes       2
Gestational Diabetes       0
Mental Health              0
Heart Rate                 2
Risk Level                18
dtype: int64

In [20]:
# drop rows where there are null values in the risk level column
df = df.dropna(subset=['Risk Level'])

numeric_features_df = df.select_dtypes(include=np.number)
num_cols = numeric_features_df.columns.tolist()

print("\nList of numeric feature names:")
print(num_cols)

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    
print(df.shape)
print(df.isnull().sum())



List of numeric feature names:
['Age', 'Systolic BP', 'Diastolic', 'BS', 'Body Temp', 'BMI', 'Previous Complications', 'Preexisting Diabetes', 'Gestational Diabetes', 'Mental Health', 'Heart Rate']
(1187, 12)
Age                       0
Systolic BP               0
Diastolic                 0
BS                        0
Body Temp                 0
BMI                       0
Previous Complications    0
Preexisting Diabetes      0
Gestational Diabetes      0
Mental Health             0
Heart Rate                0
Risk Level                0
dtype: int64


In [22]:
#check for duplicates
df.duplicated().sum()

19

In [26]:
#remove duplicates
df.drop_duplicates(inplace=True)
print(df.shape)


(1168, 12)


In [28]:
df.describe()

,Age,Systolic BP,Diastolic,BS,Body Temp,BMI,Previous Complications,Preexisting Diabetes,Gestational Diabetes,Mental Health,Heart Rate
count,1168.000000,1168.00000,1168.000000,1168.000000,1168.000000,1168.000000,1168.000000,1168.000000,1168.000000,1168.000000,1168.000000
mean,27.570205,116.83476,77.255137,7.524212,98.403253,23.355908,0.178938,0.293664,0.119863,0.340753,75.883562
std,9.245312,18.77146,14.339738,3.073423,1.097269,3.881307,0.383465,0.455635,0.324940,0.474166,7.247620
min,10.000000,70.00000,40.000000,3.000000,97.000000,0.000000,0.000000,0.000000,0.000000,0.000000,58.000000
25%,21.000000,100.00000,65.000000,6.000000,98.000000,21.000000,0.000000,0.000000,0.000000,0.000000,70.000000
50%,25.000000,120.00000,80.000000,6.900000,98.000000,23.000000,0.000000,0.000000,0.000000,0.000000,76.000000
75%,32.000000,130.00000,90.000000,8.000000,98.000000,25.025000,0.000000,1.000000,0.000000,1.000000,80.000000
max,65.000000,200.00000,140.000000,19.000000,103.000000,37.000000,1.000000,1.000000,1.000000,1.000000,92.000000


In [30]:
#outlier detection 
ranges = {
    'Age': (13, 50),
    'Systolic BP': (80, 180),
    'Diastolic': (50, 120),
    'BS': (3, 16),
    'Body Temp': (95, 102),
    'BMI': (12, 45),
    'Heart Rate': (60, 110)
}

# Replace outliers with NaN
for col, (min_val, max_val) in ranges.items():
    df.loc[(df[col] < min_val) | (df[col] > max_val), col] = np.nan

# Show how many were changed
print(df.isnull().sum())


Age                       42
Systolic BP               17
Diastolic                 14
BS                        29
Body Temp                  6
BMI                        1
Previous Complications     0
Preexisting Diabetes       0
Gestational Diabetes       0
Mental Health              0
Heart Rate                 3
Risk Level                 0
dtype: int64


In [ ]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

print(df.isnull().sum())
